# Credit Default Risk Scorecard
## Notebook 3: Model Training & Scorecard
**Author:** Simpson Gundlapally
**Model ROC-AUC: 0.8338**

Build logistic regression model, convert probabilities to credit scores, and define decision bands.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, roc_curve
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/cs-training.csv', index_col=0)
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)
df = df[df['age'] >= 18].copy()
df['RevolvingUtilizationOfUnsecuredLines'] = df['RevolvingUtilizationOfUnsecuredLines'].clip(0,1)
df['DebtRatio'] = df['DebtRatio'].clip(0,10)
df['TotalLatePays'] = (df['NumberOfTime30-59DaysPastDueNotWorse']
                      + df['NumberOfTime60-89DaysPastDueNotWorse']
                      + df['NumberOfTimes90DaysLate'])
df['DebtToIncomeRatio'] = df['DebtRatio'] * df['MonthlyIncome']
df['CreditDensity'] = df['NumberOfOpenCreditLinesAndLoans'] / (df['age'] - 17 + 0.1)
print(f'Data ready: {len(df):,} rows')

## Step 1 — Train / Test Split

In [ ]:
features = [
    'RevolvingUtilizationOfUnsecuredLines','age',
    'NumberOfTime30-59DaysPastDueNotWorse','DebtRatio',
    'MonthlyIncome','NumberOfOpenCreditLinesAndLoans',
    'NumberOfTimes90DaysLate','NumberRealEstateLoansOrLines',
    'NumberOfTime60-89DaysPastDueNotWorse','NumberOfDependents',
    'TotalLatePays','DebtToIncomeRatio','CreditDensity'
]

X = df[features]
y = df['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training set: {len(X_train):,} records ({y_train.mean()*100:.2f}% defaults)')
print(f'Test set:     {len(X_test):,} records ({y_test.mean()*100:.2f}% defaults)')

## Step 2 — Train Logistic Regression

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# class_weight='balanced' handles the 1:14 imbalance
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)

y_pred  = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:,1]
auc = roc_auc_score(y_test, y_proba)

print(f'ROC-AUC Score: {auc:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred))

## Step 3 — Convert Probability to Credit Score (300-850 scale)

In [ ]:
def prob_to_score(prob):
    """Convert default probability to FICO-style score (300-850).
    PDO (Points to Double Odds) = 20, Base score = 600 at 50:1 odds.
    """
    factor = 20 / np.log(2)
    offset = 600 - factor * np.log(50)
    score  = offset - factor * np.log(np.clip(prob / (1 - prob + 1e-10), 1e-10, None))
    return np.clip(score, 300, 850).astype(int)

results = X_test.copy()
results['ActualDefault']      = y_test.values
results['DefaultProbability'] = y_proba
results['CreditScore']        = prob_to_score(y_proba)

print('Score distribution:')
print(results['CreditScore'].describe())

results['ScoreBand'] = pd.cut(results['CreditScore'],
    bins=[299,500,600,700,850],
    labels=['300-500 DECLINE','501-600 REVIEW','601-700 CONDITIONAL','701-850 APPROVE'])

band_summary = results.groupby('ScoreBand', observed=True).agg(
    Applicants  =('ActualDefault','count'),
    Defaults    =('ActualDefault','sum'),
    DefaultRate =('ActualDefault','mean'),
    AvgScore    =('CreditScore','mean')
).reset_index()
band_summary['DefaultRate'] = (band_summary['DefaultRate']*100).round(1)
print('\nSCORECARD BAND SUMMARY:')
print(band_summary.to_string(index=False))

## Step 4 — Save Results

In [ ]:
results.to_csv('../data/scored_applicants.csv', index=False)
band_summary.to_csv('../data/band_summary.csv', index=False)
print('Results saved.')
print(f'\nFinal Model ROC-AUC: {auc:.4f}')
print('Project complete. Upload dashboard screenshots to dashboard/ folder.')